# LeetCode 346 — Moving Average from Data Stream bằng Python

**Link đề:** https://leetcode.com/problems/moving-average-from-data-stream/
**Độ khó:** Easy · **Chủ đề:** Array, Design, Queue, Data Stream · **Bài Premium**

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class MovingAverage:

    def __init__(self, size: int):
        ...

    def next(self, val: int) -> float:
        ...


# Your MovingAverage object will be instantiated and called as such:
# obj = MovingAverage(size)
# param_1 = obj.next(val)
```

## Mục tiêu học
1. Hiểu bài toán **luồng dữ liệu** (data stream): giá trị đến từng cái một, phải trả lời ngay sau mỗi giá trị.
2. Nắm **cửa sổ trượt** (sliding window) kích thước cố định và cấu trúc **hàng đợi** (queue, FIFO) với `collections.deque`.
3. Đi từ cách cộng lại cả cửa sổ mỗi lần `O(size)` tới **tổng chạy** (running sum) `O(1)`: cộng phần tử mới, trừ phần tử rời cửa sổ.
4. Viết thêm **bộ đệm vòng** (circular buffer) dùng chỉ số `% size` — cách làm trong ngôn ngữ không có deque.
5. Trace trạng thái cửa sổ, chứng minh invariant `window_sum == sum(window)`, test đối chiếu, xử lý số thực khi so sánh.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho một luồng số nguyên và kích thước cửa sổ `size`. Hãy tính **trung bình cộng của `size` giá trị gần nhất** trong luồng (nếu mới có ít hơn `size` giá trị thì lấy trung bình của tất cả giá trị đã có).

Cài đặt class `MovingAverage`:
- `MovingAverage(int size)` khởi tạo đối tượng với kích thước cửa sổ `size`.
- `double next(int val)` nhận giá trị mới `val` từ luồng và trả về trung bình của `size` giá trị cuối cùng.

**Ví dụ:**
```text
Input
["MovingAverage", "next", "next", "next", "next"]
[[3], [1], [10], [3], [5]]
Output
[null, 1.0, 5.5, 4.66667, 6.0]
```

| Lệnh | Cửa sổ (tối đa 3) | Trung bình |
|---|---|---|
| `next(1)` | `[1]` | `1 / 1 = 1.0` |
| `next(10)` | `[1, 10]` | `11 / 2 = 5.5` |
| `next(3)` | `[1, 10, 3]` | `14 / 3 ≈ 4.66667` |
| `next(5)` | `[10, 3, 5]` (số 1 rời cửa sổ) | `18 / 3 = 6.0` |

**Constraints (theo hiểu biết về đề):** `1 <= size <= 1000`; `-10^5 <= val <= 10^5`; tối đa `10^4` lần gọi `next`.

In [1]:
stream = [1, 10, 3, 5]
size = 3
for k in range(1, len(stream) + 1):
    window = stream[max(0, k - size):k]      # size phần tử cuối trong k phần tử đầu
    print(f"sau {k} giá trị: cửa sổ = {window}, trung bình = {sum(window) / len(window)}")

sau 1 giá trị: cửa sổ = [1], trung bình = 1.0
sau 2 giá trị: cửa sổ = [1, 10], trung bình = 5.5
sau 3 giá trị: cửa sổ = [1, 10, 3], trung bình = 4.666666666666667
sau 4 giá trị: cửa sổ = [10, 3, 5], trung bình = 6.0


## 2. Nền tảng Python cho bài này

### 2.1. Class cho bài Design: `__init__` lưu trạng thái, method dùng trạng thái

Đối tượng phải **nhớ** các giá trị trước đó giữa các lần gọi `next` → lưu vào thuộc tính `self.xxx`.

### 2.2. Phép chia `/` cho ra `float`
`14 / 3` cho `4.666666666666667`. LeetCode chấp nhận sai số nhỏ (thường `10^-5`), nên in `4.66667` là bản làm tròn.

In [2]:
print(14 / 3, type(14 / 3))
print(round(14 / 3, 5))
print(18 / 3)            # 6.0 — vẫn là float dù chia hết
print(14 // 3)           # 4 — chia lấy nguyên, KHÔNG dùng cho trung bình

4.666666666666667 <class 'float'>
4.66667
6.0
4


### 2.3. Queue (hàng đợi, FIFO) và `collections.deque`

**Queue**: "vào trước ra trước". Phần tử mới vào **cuối**, phần tử cũ nhất ra ở **đầu** — đúng hành vi của cửa sổ trượt.

- `deque.append(x)`: thêm vào cuối, `O(1)`.
- `deque.popleft()`: lấy ra ở đầu, `O(1)`.
- `list.pop(0)` cũng lấy ở đầu nhưng tốn `O(n)` vì phải dời mọi phần tử.

In [3]:
from collections import deque

q = deque()
for x in [1, 10, 3]:
    q.append(x)
    print("append", x, "->", q)
print("popleft ->", q.popleft(), "| còn", q)
print("len =", len(q), "| phần tử đầu q[0] =", q[0])

append 1 -> deque([1])
append 10 -> deque([1, 10])
append 3 -> deque([1, 10, 3])
popleft -> 1 | còn deque([10, 3])
len = 2 | phần tử đầu q[0] = 10


### 2.4. `deque(maxlen=k)` — tự đẩy phần tử cũ ra

Khi `deque` đã đầy (`maxlen` phần tử), `append` sẽ **tự động bỏ** phần tử ở đầu. Tiện, nhưng ta phải biết **phần tử nào bị bỏ** để trừ khỏi tổng.

In [4]:
w = deque(maxlen=3)
for x in [1, 10, 3, 5, 7]:
    dropped = w[0] if len(w) == w.maxlen else None
    w.append(x)
    print(f"append {x}: {list(w)} | bị đẩy ra: {dropped}")

append 1: [1] | bị đẩy ra: None
append 10: [1, 10] | bị đẩy ra: None
append 3: [1, 10, 3] | bị đẩy ra: None
append 5: [10, 3, 5] | bị đẩy ra: 1
append 7: [3, 5, 7] | bị đẩy ra: 10


## 3. Cách trực giác — lưu tất cả, mỗi lần cộng lại `size` phần tử cuối

Lưu mọi giá trị vào list. Mỗi lần `next`, lấy `size` phần tử cuối (`self.values[-size:]`) rồi tính `sum / len`.

In [5]:
class MovingAverageBrute:
    def __init__(self, size: int):
        self.size = size
        self.values = []

    def next(self, val: int) -> float:
        self.values.append(val)
        window = self.values[-self.size:]        # slice âm: size phần tử cuối (hoặc ít hơn)
        return sum(window) / len(window)

m = MovingAverageBrute(3)
print([m.next(v) for v in [1, 10, 3, 5]])
print([1, 2][-3:], [1, 2, 3, 4, 5][-3:])        # giải thích slice âm khi list ngắn hơn

[1.0, 5.5, 4.666666666666667, 6.0]
[1, 2] [3, 4, 5]


## 4. Nút thắt

- **Thời gian:** mỗi `next` cộng lại tới `size` phần tử → `O(size)`. Với `size = 1000` và `10^4` lần gọi → `10^7` phép cộng: chạy được nhưng lãng phí.
- **Bộ nhớ:** lưu **mọi** giá trị của luồng (`O(số lần gọi)`) dù chỉ cần `size` giá trị cuối. Với luồng "vô hạn" thì không chấp nhận được.

Quan sát: giữa hai lần gọi liên tiếp, cửa sổ chỉ thay đổi **hai** phần tử: một phần tử mới **vào**, (có thể) một phần tử cũ **ra**. Phần ở giữa giống hệt → không cần cộng lại.

In [6]:
calls, size = 10**4, 1000
print(f"Brute force: khoảng {calls * size:,} phép cộng, lưu {calls:,} giá trị")
print(f"Tổng chạy  : khoảng {calls * 2:,} phép cộng/trừ, lưu tối đa {size:,} giá trị")

Brute force: khoảng 10,000,000 phép cộng, lưu 10,000 giá trị
Tổng chạy  : khoảng 20,000 phép cộng/trừ, lưu tối đa 1,000 giá trị


## 5. Insight — tổng chạy (running sum)

Giữ biến `window_sum` = tổng các phần tử **đang trong cửa sổ**.

```text
next(val):
    thêm val vào cuối hàng đợi;   window_sum += val
    nếu hàng đợi dài hơn size:
        old = lấy phần tử đầu ra;  window_sum -= old
    return window_sum / len(hàng đợi)
```

Ví dụ `size = 3`, luồng `1, 10, 3, 5`:
- sau `3`: cửa sổ `[1, 10, 3]`, tổng `14`.
- `next(5)`: thêm 5 → tổng `19`, cửa sổ dài 4 > 3 → bỏ `1` → tổng `18`, cửa sổ `[10, 3, 5]` → `18 / 3 = 6.0` ✔.

## 6. Lời giải chính — giảng từng đoạn rồi ghép

**Đoạn 1 — `__init__` lưu 3 thứ:**
```python
self.size = size
self.window = deque()
self.window_sum = 0
```

**Đoạn 2 — đưa giá trị mới vào:**
```python
self.window.append(val)
self.window_sum += val
```

**Đoạn 3 — nếu tràn cửa sổ, bỏ phần tử cũ nhất:**
```python
if len(self.window) > self.size:
    self.window_sum -= self.window.popleft()
```
Chỉ cần `if` (không cần `while`) vì mỗi lần chỉ thêm 1 phần tử → tràn tối đa 1.

**Đoạn 4 — trả trung bình:** `return self.window_sum / len(self.window)`. Mẫu số là **số phần tử thực có**, không phải `size`.

In [7]:
class MovingAverageDeque:
    def __init__(self, size: int):
        self.size = size
        self.window = deque()
        self.window_sum = 0

    def next(self, val: int) -> float:
        self.window.append(val)
        self.window_sum += val
        if len(self.window) > self.size:
            self.window_sum -= self.window.popleft()
        return self.window_sum / len(self.window)

m = MovingAverageDeque(3)
print([m.next(v) for v in [1, 10, 3, 5]])

[1.0, 5.5, 4.666666666666667, 6.0]


## 7. TRACE từng bước

Bảng tay `size = 3`, luồng `1, 10, 3, 5, -4, 2`:

| next(val) | cửa sổ sau khi thêm | tràn? bỏ | window_sum | len | trung bình |
|---:|---|---|---:|---:|---:|
| 1 | [1] | — | 1 | 1 | 1.0 |
| 10 | [1, 10] | — | 11 | 2 | 5.5 |
| 3 | [1, 10, 3] | — | 14 | 3 | 4.66667 |
| 5 | [1, 10, 3, 5] | bỏ 1 | 18 | 3 | 6.0 |
| -4 | [10, 3, 5, -4] | bỏ 10 | 4 | 3 | 1.33333 |
| 2 | [3, 5, -4, 2] | bỏ 3 | 3 | 3 | 1.0 |

Ý nghĩa: cột "tràn? bỏ" là phần tử rời cửa sổ; `window_sum` luôn bằng tổng cửa sổ **sau** khi bỏ.

In [8]:
def trace_moving_average(size: int, stream: list[int]) -> list[float]:
    window = deque()
    window_sum = 0
    out = []
    for val in stream:
        window.append(val)
        window_sum += val
        dropped = None
        if len(window) > size:
            dropped = window.popleft()
            window_sum -= dropped
        avg = window_sum / len(window)
        out.append(avg)
        print(f"next({val:>3}) | bỏ = {str(dropped):>4} | cửa sổ = {list(window)!s:<14} | sum = {window_sum:>3} | len = {len(window)} | avg = {avg:.5f}")
        assert window_sum == sum(window)     # kiểm tra invariant ngay trong trace
    return out

trace_moving_average(3, [1, 10, 3, 5, -4, 2])

next(  1) | bỏ = None | cửa sổ = [1]            | sum =   1 | len = 1 | avg = 1.00000
next( 10) | bỏ = None | cửa sổ = [1, 10]        | sum =  11 | len = 2 | avg = 5.50000
next(  3) | bỏ = None | cửa sổ = [1, 10, 3]     | sum =  14 | len = 3 | avg = 4.66667
next(  5) | bỏ =    1 | cửa sổ = [10, 3, 5]     | sum =  18 | len = 3 | avg = 6.00000
next( -4) | bỏ =   10 | cửa sổ = [3, 5, -4]     | sum =   4 | len = 3 | avg = 1.33333
next(  2) | bỏ =    3 | cửa sổ = [5, -4, 2]     | sum =   3 | len = 3 | avg = 1.00000


[1.0, 5.5, 4.666666666666667, 6.0, 1.3333333333333333, 1.0]

In [9]:
trace_moving_average(1, [7, -2, 9])     # size = 1: trung bình chính là giá trị vừa đến

next(  7) | bỏ = None | cửa sổ = [7]            | sum =   7 | len = 1 | avg = 7.00000
next( -2) | bỏ =    7 | cửa sổ = [-2]           | sum =  -2 | len = 1 | avg = -2.00000
next(  9) | bỏ =   -2 | cửa sổ = [9]            | sum =   9 | len = 1 | avg = 9.00000


[7.0, -2.0, 9.0]

## 8. Các lời giải khác

### Cách 3 — `deque(maxlen=size)` + tổng chạy
Phải lấy phần tử sắp bị đẩy ra **trước khi** `append`.

In [10]:
class MovingAverageMaxlen:
    def __init__(self, size: int):
        self.window = deque(maxlen=size)
        self.window_sum = 0

    def next(self, val: int) -> float:
        if len(self.window) == self.window.maxlen:
            self.window_sum -= self.window[0]      # phần tử sắp bị đẩy ra
        self.window.append(val)
        self.window_sum += val
        return self.window_sum / len(self.window)

m = MovingAverageMaxlen(3)
print([round(m.next(v), 5) for v in [1, 10, 3, 5]])

[1.0, 5.5, 4.66667, 6.0]


### Cách 4 — bộ đệm vòng (circular buffer)

Mảng cố định dài `size` và biến `count` (tổng số giá trị đã nhận). Giá trị thứ `count` (đếm từ 0) được ghi vào ô `count % size` — **ghi đè** đúng giá trị cũ nhất. Không cần deque, chỉ cần list và phép `%`.

Ví dụ `size = 3`: các giá trị thứ 0, 1, 2, 3, 4 lần lượt vào ô `0, 1, 2, 0, 1`.

In [11]:
class MovingAverageCircular:
    def __init__(self, size: int):
        self.size = size
        self.buffer = [0] * size
        self.count = 0
        self.window_sum = 0

    def next(self, val: int) -> float:
        idx = self.count % self.size
        self.window_sum += val - self.buffer[idx]   # ô chưa dùng chứa 0 nên trừ 0 vô hại
        self.buffer[idx] = val
        self.count += 1
        return self.window_sum / min(self.count, self.size)

for c in range(6):
    print("giá trị thứ", c, "-> ô", c % 3)
m = MovingAverageCircular(3)
print([round(m.next(v), 5) for v in [1, 10, 3, 5]], "| buffer =", m.buffer)

giá trị thứ 0 -> ô 0
giá trị thứ 1 -> ô 1
giá trị thứ 2 -> ô 2
giá trị thứ 3 -> ô 0
giá trị thứ 4 -> ô 1
giá trị thứ 5 -> ô 2
[1.0, 5.5, 4.66667, 6.0] | buffer = [5, 10, 3]


### Cách 5 — list + mảng tổng tiền tố (nối với bài 303)

Lưu `prefix` của luồng; trung bình cửa sổ = `(prefix[k] - prefix[max(0, k - size)]) / min(k, size)`. `O(1)` thời gian mỗi lần, nhưng bộ nhớ `O(số lần gọi)`.

In [12]:
class MovingAveragePrefix:
    def __init__(self, size: int):
        self.size = size
        self.prefix = [0]

    def next(self, val: int) -> float:
        self.prefix.append(self.prefix[-1] + val)
        k = len(self.prefix) - 1                 # số giá trị đã nhận
        start = max(0, k - self.size)
        return (self.prefix[k] - self.prefix[start]) / (k - start)

m = MovingAveragePrefix(3)
print([round(m.next(v), 5) for v in [1, 10, 3, 5]], "| prefix =", m.prefix)

[1.0, 5.5, 4.66667, 6.0] | prefix = [0, 1, 11, 14, 19]


## 9. Code nộp LeetCode

### Cách chính — deque + tổng chạy

In [13]:
class MovingAverage:

    def __init__(self, size: int):
        self.size = size
        self.window = deque()
        self.window_sum = 0

    def next(self, val: int) -> float:
        self.window.append(val)
        self.window_sum += val
        if len(self.window) > self.size:
            self.window_sum -= self.window.popleft()
        return self.window_sum / len(self.window)


# Your MovingAverage object will be instantiated and called as such:
obj = MovingAverage(3)
print([obj.next(v) for v in [1, 10, 3, 5]])

[1.0, 5.5, 4.666666666666667, 6.0]


Lưu ý: trên LeetCode, `deque` cần `from collections import deque` (môi trường LeetCode thường import sẵn, nhưng viết rõ ra trên đầu file là an toàn).

### Cách nộp thay thế — bộ đệm vòng (không cần import)

In [14]:
class MovingAverageArray:

    def __init__(self, size: int):
        self.size = size
        self.buffer = [0] * size
        self.count = 0
        self.window_sum = 0

    def next(self, val: int) -> float:
        idx = self.count % self.size
        self.window_sum += val - self.buffer[idx]
        self.buffer[idx] = val
        self.count += 1
        return self.window_sum / min(self.count, self.size)

obj = MovingAverageArray(3)
print([obj.next(v) for v in [1, 10, 3, 5]])

[1.0, 5.5, 4.666666666666667, 6.0]


## 10. Vì sao đúng?

**Invariant (sau mỗi lần `next` trả về):**
1. `window` chứa đúng `min(k, size)` giá trị **mới nhất** trong `k` giá trị đã nhận, theo thứ tự đến.
2. `window_sum == sum(window)`.

**Chứng minh quy nạp theo số lần gọi `k`:**
- `k = 0`: cửa sổ rỗng, tổng 0 ✔.
- Từ `k` lên `k + 1`: thêm `val` vào cuối và cộng vào tổng → cửa sổ chứa `min(k, size) + 1` giá trị mới nhất, tổng đúng. Nếu số này vượt `size` (tức bằng `size + 1`), phần tử ở đầu là phần tử **cũ nhất**, bỏ nó và trừ khỏi tổng → còn `size` giá trị mới nhất, tổng vẫn đúng ✔.

Vì vậy `window_sum / len(window)` đúng là trung bình yêu cầu.

**Bộ đệm vòng:** ô `count % size` luôn giữ giá trị **cũ nhất** khi cửa sổ đã đầy (vì các giá trị được ghi xoay vòng theo thứ tự), nên `val - buffer[idx]` = "cộng mới, trừ cũ". Khi chưa đầy, ô đó còn 0 nên chỉ cộng mới.

## 11. Time / Space Complexity và so sánh

Gọi `k` = số lần gọi `next`, `size` = kích thước cửa sổ.

| Cách | Mỗi `next` | Bộ nhớ | Ghi chú |
|---|---|---|---|
| Lưu tất cả + `sum(slice)` | `O(size)` | `O(k)` | dễ nghĩ |
| **deque + tổng chạy** | **`O(1)`** | **`O(size)`** | chuẩn |
| `deque(maxlen)` + tổng chạy | `O(1)` | `O(size)` | phải lấy phần tử sắp rơi trước |
| Bộ đệm vòng | `O(1)` | `O(size)` | không cần deque |
| Prefix sum | `O(1)` | `O(k)` | nối với bài 303 |

**Ghi chú số thực:** tổng chạy dùng **số nguyên** (`val` là int) nên không tích lũy sai số; chỉ phép chia cuối cùng mới tạo float. Nếu `val` là số thực, cộng/trừ liên tục có thể tích lũy sai số nhỏ.

## 12. Bộ test — mẫu đề, biên, đối chiếu

Kết quả là `float` → so sánh với **dung sai** bằng `math.isclose` thay vì `==`.

In [15]:
import math

def run_ops(cls, commands, args):
    out, obj = [], None
    for cmd, arg in zip(commands, args):
        if obj is None:
            obj = cls(*arg)
            out.append(None)
        else:
            out.append(getattr(obj, cmd)(*arg))
    return out

def close_list(a, b, tol=1e-5):
    return len(a) == len(b) and all((x is None and y is None) or (x is not None and y is not None and math.isclose(x, y, abs_tol=tol)) for x, y in zip(a, b))

def check(name, actual, expected):
    assert close_list(actual, expected), (name, actual, expected)
    print(f"PASS {name}: {[None if x is None else round(x, 5) for x in actual]}")

check("ví dụ đề", run_ops(MovingAverage, ["MovingAverage", "next", "next", "next", "next"], [[3], [1], [10], [3], [5]]),
      [None, 1.0, 5.5, 4.66667, 6.0])
check("size = 1", run_ops(MovingAverage, ["MovingAverage", "next", "next"], [[1], [4], [-6]]), [None, 4.0, -6.0])
check("size lớn hơn số lần gọi", run_ops(MovingAverage, ["MovingAverage", "next", "next"], [[100], [2], [4]]), [None, 2.0, 3.0])
check("số âm", run_ops(MovingAverage, ["MovingAverage", "next", "next", "next"], [[2], [-5], [-5], [10]]), [None, -5.0, -5.0, 2.5])
check("biên giá trị", run_ops(MovingAverage, ["MovingAverage", "next", "next"], [[2], [10**5], [-10**5]]), [None, 1e5, 0.0])
m = MovingAverage(1000)
for i in range(10**4):
    last = m.next(i)
check("10^4 lần gọi, size 1000", [last], [sum(range(9000, 10000)) / 1000])

PASS ví dụ đề: [None, 1.0, 5.5, 4.66667, 6.0]
PASS size = 1: [None, 4.0, -6.0]
PASS size lớn hơn số lần gọi: [None, 2.0, 3.0]
PASS số âm: [None, -5.0, -5.0, 2.5]
PASS biên giá trị: [None, 100000.0, 0.0]
PASS 10^4 lần gọi, size 1000: [9499.5]


In [16]:
import random
random.seed(346)
classes = [MovingAverageBrute, MovingAverageDeque, MovingAverageMaxlen, MovingAverageCircular,
           MovingAveragePrefix, MovingAverage, MovingAverageArray]
for _ in range(300):
    size = random.randint(1, 8)
    stream = [random.randint(-10**5, 10**5) for _ in range(random.randint(1, 40))]
    objs = [cls(size) for cls in classes]
    for k, val in enumerate(stream, start=1):
        window = stream[max(0, k - size):k]
        expected = sum(window) / len(window)
        for o in objs:
            assert math.isclose(o.next(val), expected, abs_tol=1e-9), (type(o).__name__, size, stream[:k])
print("300 luồng ngẫu nhiên × 7 cách: đều khớp với định nghĩa")

300 luồng ngẫu nhiên × 7 cách: đều khớp với định nghĩa


## 13. Lỗi người mới hay gặp

### Lỗi 1: Chia cho `size` thay vì số phần tử hiện có
Ở những lần gọi đầu (cửa sổ chưa đầy), kết quả bị kéo nhỏ.

In [17]:
class WrongDivide(MovingAverageDeque):
    def next(self, val):
        super().next(val)                          # cập nhật cửa sổ như bản đúng
        return self.window_sum / self.size         # SAI: chưa đầy vẫn chia size

w, ok = WrongDivide(3), MovingAverageDeque(3)
print("sai :", [round(w.next(v), 5) for v in [1, 10, 3, 5]])
print("đúng:", [round(ok.next(v), 5) for v in [1, 10, 3, 5]])

sai : [0.33333, 3.66667, 4.66667, 6.0]
đúng: [1.0, 5.5, 4.66667, 6.0]


### Lỗi 2: Dùng `//` thay vì `/`
`11 // 2 = 5` thay vì `5.5`.

### Lỗi 3: `deque(maxlen)` nhưng quên trừ phần tử bị đẩy ra
Tổng chạy cứ tăng mãi.

In [18]:
class WrongMaxlen:
    def __init__(self, size):
        self.window = deque(maxlen=size)
        self.window_sum = 0
    def next(self, val):
        self.window.append(val)                    # SAI: phần tử bị đẩy ra không được trừ
        self.window_sum += val
        return self.window_sum / len(self.window)

w = WrongMaxlen(3)
print("sai :", [round(w.next(v), 5) for v in [1, 10, 3, 5]], "| đúng: [1.0, 5.5, 4.66667, 6.0]")
print("// :", 11 // 2, "| / :", 11 / 2)

sai : [1.0, 5.5, 4.66667, 6.33333] | đúng: [1.0, 5.5, 4.66667, 6.0]
// : 5 | / : 5.5


### Lỗi 4: Dùng `list.pop(0)` cho hàng đợi
Đúng kết quả nhưng mỗi lần `O(size)`. Đo thử:

In [19]:
import time

class ListQueue:
    def __init__(self, size):
        self.size, self.window, self.window_sum = size, [], 0
    def next(self, val):
        self.window.append(val)
        self.window_sum += val
        if len(self.window) > self.size:
            self.window_sum -= self.window.pop(0)  # O(size) mỗi lần
        return self.window_sum / len(self.window)

for cls in [ListQueue, MovingAverage]:
    o = cls(50000)
    t0 = time.perf_counter()
    for i in range(100000):
        o.next(i)
    print(f"{cls.__name__:<14} ≈ {time.perf_counter() - t0:.3f}s")

ListQueue      ≈ 0.477s
MovingAverage  ≈ 0.019s


### Lỗi 5: Dùng biến lớp (class attribute) thay vì `self.`
Khai báo `window = deque()` ngay trong thân class → **mọi đối tượng dùng chung** một deque!

In [20]:
class SharedState:
    window = deque()               # SAI: thuộc về class, dùng chung
    def __init__(self, size):
        self.size = size
    def next(self, val):
        self.window.append(val)
        if len(self.window) > self.size:
            self.window.popleft()
        return sum(self.window) / len(self.window)

a, c = SharedState(3), SharedState(3)
a.next(100)
print("đối tượng c mới tạo nhưng next(1) =", c.next(1), "(đúng phải là 1.0)")

đối tượng c mới tạo nhưng next(1) = 50.5 (đúng phải là 1.0)


## 14. Mẹo nhận dạng pattern

- **"Data stream"**, **"k phần tử gần nhất"**, **"cửa sổ kích thước cố định"** → **hàng đợi + tổng chạy** (sliding window).
- Cửa sổ trượt kích thước cố định luôn có nhịp: **thêm phần tử mới → nếu tràn thì bỏ phần tử cũ → cập nhật đáp án**.
- Cần lấy ra ở đầu, thêm vào ở cuối → `collections.deque`, không phải `list.pop(0)`.
- Ngôn ngữ/ràng buộc không có deque → **circular buffer** với `% size`.
- Bài liên quan: Maximum Average Subarray I (643), Number of Recent Calls (933 — cửa sổ theo **thời gian**), Logger Rate Limiter (359), Design Hit Counter (362).

## 15. Biến thể

### Biến thể 1 — Maximum Average Subarray I (LeetCode 643)
Cho mảng, tìm trung bình lớn nhất của đoạn con dài đúng `k`. Dùng cùng kỹ thuật tổng chạy.

### Biến thể 2 — Number of Recent Calls (LeetCode 933)
Cửa sổ theo **thời gian** `[t - 3000, t]`: bỏ ở đầu **trong khi** (`while`) phần tử quá cũ — số phần tử bỏ mỗi lần có thể > 1.

In [21]:
def find_max_average(nums: list[int], k: int) -> float:
    window_sum = sum(nums[:k])
    best = window_sum
    for i in range(k, len(nums)):
        window_sum += nums[i] - nums[i - k]      # vào nums[i], ra nums[i - k]
        best = max(best, window_sum)
    return best / k

class RecentCounter:
    def __init__(self):
        self.q = deque()
    def ping(self, t: int) -> int:
        self.q.append(t)
        while self.q[0] < t - 3000:              # while: có thể bỏ nhiều phần tử
            self.q.popleft()
        return len(self.q)

print(find_max_average([1, 12, -5, -6, 50, 3], 4))      # 12.75
rc = RecentCounter()
print([rc.ping(t) for t in [1, 100, 3001, 3002]])       # [1, 2, 3, 3]

12.75
[1, 2, 3, 3]


## 16. Bài tập tự luyện (TODO — không chạy sẵn)

### Bài 1 — Tính tay
`size = 2`, luồng `4, 8, -2, 6`. Lập bảng (cửa sổ, sum, trung bình) sau mỗi `next`.

### Bài 2 — Dự đoán output
```python
d = deque(maxlen=2)
for x in "abc":
    d.append(x)
print(list(d), 7 % 3, 3 % 3)
```

### Bài 3 — Điền chỗ trống

In [22]:
class ExerciseMovingAverage:
    def __init__(self, size: int):
        self.size = size
        self.window = deque()
        self.window_sum = 0

    def next(self, val: int) -> float:
        # TODO 1: thêm val vào window và cộng vào window_sum
        # TODO 2: nếu tràn, popleft và trừ khỏi window_sum
        # TODO 3: trả trung bình
        pass

# m = ExerciseMovingAverage(3)
# assert [round(m.next(v), 5) for v in [1, 10, 3, 5]] == [1.0, 5.5, 4.66667, 6.0]

### Bài 4 — Tự viết: moving **max** đơn giản
Viết `ExerciseMovingMax(size)` với `next(val)` trả về **giá trị lớn nhất** trong `size` giá trị gần nhất. Cho phép `O(size)` mỗi lần (dùng `max(self.window)`).

In [23]:
class ExerciseMovingMax:
    def __init__(self, size: int):
        # TODO
        pass

    def next(self, val: int) -> int:
        # TODO
        pass

# m = ExerciseMovingMax(2)
# assert [m.next(v) for v in [3, 1, 2, 5, 4]] == [3, 3, 2, 5, 5]

### Bài 5 — Biến thể: trung bình của **mọi** đoạn dài k trong mảng
Trả về list các trung bình (giống đầu ra của MovingAverage nhưng chỉ khi cửa sổ đã đủ `k`).

In [24]:
def exercise_all_window_averages(nums: list[int], k: int) -> list[float]:
    # TODO: tổng chạy, chỉ append khi i >= k - 1
    pass

# assert exercise_all_window_averages([1, 3, 2, 6, -1, 4, 1, 8, 2], 5) == [2.2, 2.8, 2.4, 3.6, 2.8]

### Bài 6 — Mini challenge: Moving Average bằng circular buffer, không dùng `count` vô hạn
Bộ đệm vòng ở mục 8 dùng `count` tăng mãi. Hãy viết lại với hai biến `head` (vị trí sẽ ghi tiếp, luôn trong `[0, size)`) và `filled` (số ô đã dùng, tối đa `size`).

In [25]:
class ExerciseCircular:
    def __init__(self, size: int):
        # TODO: buffer, head, filled, window_sum
        pass

    def next(self, val: int) -> float:
        # TODO: nếu filled == size thì trừ buffer[head]; ghi val; head = (head + 1) % size
        pass

# m = ExerciseCircular(3)
# assert [round(m.next(v), 5) for v in [1, 10, 3, 5, -4, 2]] == [1.0, 5.5, 4.66667, 6.0, 1.33333, 1.0]

## 17. Gợi ý 3 mức

**Bài 1** — (1) Cửa sổ tối đa 2 phần tử. (2) Từ lần gọi thứ 3, mỗi lần bỏ 1 phần tử. (3) Trung bình lần lượt `4.0, 6.0, 3.0, 2.0`.

**Bài 2** — (1) `maxlen=2` giữ 2 phần tử cuối. (2) `7 = 2·3 + 1`. (3) `3 % 3 = 0`.

**Bài 3** — (1) `append` + `+=`. (2) `if len(...) > size`. (3) Chia cho `len(self.window)`.

**Bài 4** — (1) Giống bài chính nhưng không cần tổng. (2) `deque(maxlen=size)`. (3) `return max(self.window)`.

**Bài 5** — (1) Tính tổng `k` phần tử đầu. (2) Trượt: `+ nums[i] - nums[i - k]`. (3) So sánh float trong test: các giá trị này chia hết đẹp nên `==` vẫn đúng, nhưng nên dùng `round`.

**Bài 6** — (1) `filled` tăng tới `size` rồi dừng. (2) Khi đầy, ô `head` chứa giá trị cũ nhất. (3) Mẫu số là `filled`.

## 18. Đáp án tham khảo

**Bài 1:**

| next | cửa sổ | sum | trung bình |
|---:|---|---:|---:|
| 4 | [4] | 4 | 4.0 |
| 8 | [4, 8] | 12 | 6.0 |
| -2 | [8, -2] | 6 | 3.0 |
| 6 | [-2, 6] | 4 | 2.0 |

**Bài 2:** `['b', 'c'] 1 0`.

In [26]:
class AnswerMovingAverage:
    def __init__(self, size: int):
        self.size = size
        self.window = deque()
        self.window_sum = 0

    def next(self, val: int) -> float:
        self.window.append(val)
        self.window_sum += val
        if len(self.window) > self.size:
            self.window_sum -= self.window.popleft()
        return self.window_sum / len(self.window)

class AnswerMovingMax:
    def __init__(self, size: int):
        self.window = deque(maxlen=size)

    def next(self, val: int) -> int:
        self.window.append(val)
        return max(self.window)

def answer_all_window_averages(nums: list[int], k: int) -> list[float]:
    result = []
    window_sum = 0
    for i, x in enumerate(nums):
        window_sum += x
        if i >= k:
            window_sum -= nums[i - k]
        if i >= k - 1:
            result.append(window_sum / k)
    return result

class AnswerCircular:
    def __init__(self, size: int):
        self.size = size
        self.buffer = [0] * size
        self.head = 0
        self.filled = 0
        self.window_sum = 0

    def next(self, val: int) -> float:
        if self.filled == self.size:
            self.window_sum -= self.buffer[self.head]
        else:
            self.filled += 1
        self.buffer[self.head] = val
        self.window_sum += val
        self.head = (self.head + 1) % self.size
        return self.window_sum / self.filled

m = AnswerMovingAverage(2)
assert [m.next(v) for v in [4, 8, -2, 6]] == [4.0, 6.0, 3.0, 2.0]
d = deque(maxlen=2)
for x in "abc":
    d.append(x)
assert (list(d), 7 % 3, 3 % 3) == (["b", "c"], 1, 0)
m = AnswerMovingAverage(3)
assert [round(m.next(v), 5) for v in [1, 10, 3, 5]] == [1.0, 5.5, 4.66667, 6.0]
m = AnswerMovingMax(2)
assert [m.next(v) for v in [3, 1, 2, 5, 4]] == [3, 3, 2, 5, 5]
assert [round(x, 5) for x in answer_all_window_averages([1, 3, 2, 6, -1, 4, 1, 8, 2], 5)] == [2.2, 2.8, 2.4, 3.6, 2.8]
m = AnswerCircular(3)
assert [round(m.next(v), 5) for v in [1, 10, 3, 5, -4, 2]] == [1.0, 5.5, 4.66667, 6.0, 1.33333, 1.0]
random.seed(7)
for _ in range(200):
    size = random.randint(1, 6)
    a, c = AnswerCircular(size), MovingAverage(size)
    for v in [random.randint(-50, 50) for _ in range(30)]:
        assert math.isclose(a.next(v), c.next(v))
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
from collections import deque

class MovingAverage:
    def __init__(self, size: int):
        self.size = size
        self.window = deque()
        self.window_sum = 0

    def next(self, val: int) -> float:
        self.window.append(val)
        self.window_sum += val
        if len(self.window) > self.size:
            self.window_sum -= self.window.popleft()
        return self.window_sum / len(self.window)
```

**Template cửa sổ trượt cố định:** thêm phần tử mới (cập nhật tổng) → nếu tràn, bỏ phần tử cũ nhất (cập nhật tổng) → tính đáp án từ tổng.

**Checklist trước khi submit:**
- [ ] Trạng thái lưu bằng `self.` trong `__init__` (không dùng biến lớp)?
- [ ] Chia cho `len(self.window)` (số phần tử thực có), không phải `size`?
- [ ] Dùng `/` (float), không phải `//`?
- [ ] Bỏ phần tử cũ bằng `popleft()` của `deque` (không `list.pop(0)`)?
- [ ] Nếu dùng `deque(maxlen)`: đã trừ phần tử bị đẩy ra **trước** khi `append`?
- [ ] Có `from collections import deque`; tên class `MovingAverage`, method `next(self, val)`?